### Imports

In [1]:
import os, sys
import numpy as np
import yaml
from tqdm import tqdm
from PIL import Image

# Add the project root directory to sys.path
project_root = os.path.abspath('..')  # Adjust this path if needed
sys.path.append(project_root)

from utils import load_images

from eval_segmentation import write_seg_metrics_csv, write_avg_metrics_yaml_from_csv




2024-08-19 10:08:19.214115: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-19 10:08:19.228800: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-19 10:08:19.233276: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-08-19 10:08:19.246404: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-19 10:08:20.446822: W tensorflow/compiler/tf2

[08/19/24 10:08:24] WARNING  Your inference package version 0.12.1 is out of date! Please upgrade to ]8;id=323343;file:///usr/local/lib/python3.10/site-packages/inference/core/__init__.py\__init__.py]8;;\:]8;id=819981;file:///usr/local/lib/python3.10/site-packages/inference/core/__init__.py#35\35]8;;\
                             version 0.16.2 of inference for the latest features and bug fixes by                  
                             running `pip install --upgrade inference`.                                            

### Datasets

In [2]:
# Load the YAML file into a dictionary
dataset_yaml_file = '/workspace/home/data/datasets.yaml'
with open(dataset_yaml_file, 'r') as file:
    datasets = yaml.safe_load(file)

#datasets

## Evaluate data
### Load data


In [3]:
# get dataset
dataset_key = "ISIC2019"#'Pascal_VOC07' #
dataset = datasets[dataset_key]

# load the ground truth, predicted segmentations and images
segmentation_path = f"/workspace/home/data/{datasets[dataset_key]['dataset_dir']}/seg"
segmentation_gt_path = datasets[dataset_key]['gt_path']

segmentation_masks_cofounder, _ = load_images(dir_path=segmentation_path, confounder_dir=dataset['confounder_dir'], no_confounder_dir=dataset['no_confounder_dir'])
segmentation_masks_gt = load_images(dir_path=segmentation_gt_path)

### Calculate metrics

In [4]:
# create csv path
timestamp = 'no_timestamp_yet'
base_dir = f"/workspace/home/pipeline_outputs/{datasets[dataset_key]['dataset_dir']}/{timestamp}"
os.makedirs(base_dir, exist_ok=True)
csv_path = os.path.join(base_dir, 'seg_metrics.csv')

# create csv file
dilation_kernel_size = 5
dilation_iterations = 4
dilation_function_params = {'kernel_size': dilation_kernel_size, 'iterations': dilation_iterations}
write_seg_metrics_csv(csv_path, segmentation_masks_cofounder, segmentation_masks_gt, dilation_function_params)

# create yaml file
yaml_path = os.path.join(base_dir, 'seg_metrics_avg.yaml')
write_avg_metrics_yaml_from_csv(csv_path, yaml_path)

